## Importation des bibliothèques

In [ ]:
import pandas as pd
from great_tables import GT
import matplotlib.pyplot as plt
import geopandas as gpd
import requests
import io

## Importation de la base de données

In [ ]:
# Récupération des données via l'url accessible sur : https://www.data.gouv.fr/datasets/recensement-des-equipements-sportifs-espaces-et-sites-de-pratiques
# Chargement long

# data = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/ea4f5879-af40-4e3e-949d-812d6eeb5e02")

# Récupération directement depuis le dernier csv téléchargé au préalable et stocké sur le s3

data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")
data.head(10)

## Analyses descriptives

In [ ]:
total_equipements = data.shape[0]
print(total_equipements)

### Répartition par type d'équipements

In [ ]:
equip_types = (
    data['equip_type_famille']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

equip_types.head(10).plot(kind='bar')
plt.title("Top 10 des types d'équipements (%)")
plt.ylabel("Part (%)")
plt.show()

In [ ]:
equip_dep = (
    data
    .groupby('dep_nom')
    .size()
    .sort_values(ascending=False)
)

equip_dep.head(10).plot(kind='bar')
plt.title("Top 10 départements les plus équipés")
plt.ylabel("Nombre d'équipements")
plt.show()

### Génération d'une carte qui répértorie le nombre d'équipement par famille d'équipement

In [ ]:
def plot_carte_nb_equipement_par_famille(famille=None):
    new_data = data.copy()
    if famille is not None:
        new_data = data[data['equip_type_famille'] == famille]
    equip_dep = (
    new_data
    .groupby('dep_nom')
    .size()
    .sort_values(ascending=False)
    )

    #1. Conversion de la Series en DataFrame
    df = equip_dep.reset_index()
    df.columns = ["departement", "valeur"]

    #2. Téléchargement du GeoJSON
    url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"
    response = requests.get(url)
    gdf = gpd.read_file(io.BytesIO(response.content))

    #3. Fusion
    gdf = gdf.merge(df, left_on="nom", right_on="departement", how="left")

    #4. Carte 
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))

    gdf.plot(
        column="valeur",
        ax=ax,
        cmap="YlOrRd",
        legend=True,
        legend_kwds={
            "label": "Nombre d'équipements",
            "orientation": "vertical",
            "shrink": 0.6,
        },
        missing_kwds={
            "color": "#e0e0e0",
            "label": "Données manquantes"
        },
        edgecolor="white",
        linewidth=0.5,
        vmin=df["valeur"].min(),
        vmax=df["valeur"].max()
    )

    ax.set_title("Nombre d'équipements par département", fontsize=16, fontweight="bold", pad=15)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_carte_nb_equipement_par_famille()
